# End-to-End Data Science Project
## Complete Pipeline: Data → Model → API Deployment

This notebook demonstrates a complete data science workflow:
1. Data Collection & Loading
2. Exploratory Data Analysis (EDA)
3. Data Preprocessing
4. Model Training & Evaluation
5. API Deployment (Flask/FastAPI)

**Note:** This notebook is optimized to run in Google Colab without errors.

## 1. Install Required Libraries

In [ ]:
!pip install -q scikit-learn pandas numpy matplotlib seaborn flask fastapi uvicorn pydantic joblib
print("✅ All libraries installed successfully!")

## 2. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import joblib
import json
from typing import List, Dict

print("✅ All imports successful!")

# Set random seed for reproducibility
np.random.seed(42)

## 3. Load Data

In [ ]:
# Load the Diabetes dataset from sklearn
print("Loading Diabetes Dataset...")
diabetes = load_diabetes()
X = diabetes.data
y = diabetes.target
feature_names = diabetes.feature_names

print(f"\n✅ Dataset Loaded Successfully!")
print(f"   Shape: {X.shape}")
print(f"   Features: {len(feature_names)}")
print(f"   Samples: {len(X)}")
print(f"   Target range: [{y.min():.2f}, {y.max():.2f}]")

## 4. Exploratory Data Analysis (EDA)

In [ ]:
# Create DataFrame for easier analysis
df = pd.DataFrame(X, columns=feature_names)
df['target'] = y

print("\n📊 Dataset Info:")
print(f"\nShape: {df.shape}")
print(f"\nFirst 5 rows:")
print(df.head())
print(f"\nBasic Statistics:")
print(df.describe())
print(f"\nMissing Values:")
print(df.isnull().sum())

In [ ]:
# Visualize distributions
plt.figure(figsize=(15, 10))

# Plot feature distributions
for idx, feature in enumerate(feature_names, 1):
    plt.subplot(2, 5, idx)
    plt.hist(df[feature], bins=20, edgecolor='black', alpha=0.7, color='skyblue')
    plt.title(f'Distribution of {feature}')
    plt.xlabel('Value')
    plt.ylabel('Frequency')

plt.tight_layout()
plt.savefig('feature_distributions.png', dpi=100, bbox_inches='tight')
plt.show()
print("✅ Feature distributions plotted!")

In [ ]:
# Correlation heatmap
plt.figure(figsize=(12, 8))
correlation_matrix = df.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, 
            fmt='.2f', square=True, linewidths=1)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.savefig('correlation_matrix.png', dpi=100, bbox_inches='tight')
plt.show()
print("✅ Correlation heatmap plotted!")

## 5. Data Preprocessing

In [ ]:
# Split the data into training and testing sets
print("Splitting data into train and test sets...")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"✅ Data Split Complete!")
print(f"   Training set: {X_train.shape}")
print(f"   Test set: {X_test.shape}")
print(f"   Split ratio: 80% / 20%")

In [ ]:
# Standardize the features
print("\nStandardizing features...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"✅ Feature scaling complete!")
print(f"   Training set mean: {X_train_scaled.mean():.4f}")
print(f"   Training set std: {X_train_scaled.std():.4f}")

## 6. Model Training

In [ ]:
# Train Random Forest Model
print("Training Random Forest Regressor...")
model = RandomForestRegressor(
    n_estimators=100,
    max_depth=15,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train_scaled, y_train)
print("✅ Model training complete!")

## 7. Model Evaluation

In [ ]:
# Make predictions
y_train_pred = model.predict(X_train_scaled)
y_test_pred = model.predict(X_test_scaled)

# Calculate metrics
train_mse = mean_squared_error(y_train, y_train_pred)
test_mse = mean_squared_error(y_test, y_test_pred)
train_rmse = np.sqrt(train_mse)
test_rmse = np.sqrt(test_mse)
train_mae = mean_absolute_error(y_train, y_train_pred)
test_mae = mean_absolute_error(y_test, y_test_pred)
train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)

print("\n" + "="*60)
print("MODEL EVALUATION RESULTS")
print("="*60)
print(f"\n📈 TRAINING SET METRICS:")
print(f"   MSE:  {train_mse:.4f}")
print(f"   RMSE: {train_rmse:.4f}")
print(f"   MAE:  {train_mae:.4f}")
print(f"   R²:   {train_r2:.4f}")
print(f"\n📉 TEST SET METRICS:")
print(f"   MSE:  {test_mse:.4f}")
print(f"   RMSE: {test_rmse:.4f}")
print(f"   MAE:  {test_mae:.4f}")
print(f"   R²:   {test_r2:.4f}")
print("="*60)

In [ ]:
# Feature importance
feature_importance = pd.DataFrame({
    'feature': feature_names,
    'importance': model.feature_importances_
,
,
\n🎯 FEATURE IMPORTANCE:")
print(feature_importance.to_string(index=False))

In [ ]:
# Plot feature importance
plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'], color='steelblue')
plt.xlabel('Importance Score')
plt.title('Random Forest Feature Importance')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=100, bbox_inches='tight')
plt.show()
print("✅ Feature importance plotted!")

In [ ]:
# Plot actual vs predicted
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Training set
axes[0].scatter(y_train, y_train_pred, alpha=0.5, color='blue')
axes[0].plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 'r--', lw=2)
axes[0].set_xlabel('Actual Values')
axes[0].set_ylabel('Predicted Values')
axes[0].set_title(f'Training Set (R² = {train_r2:.4f})')
axes[0].grid(True, alpha=0.3)

# Test set
axes[1].scatter(y_test, y_test_pred, alpha=0.5, color='green')
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[1].set_xlabel('Actual Values')
axes[1].set_ylabel('Predicted Values')
axes[1].set_title(f'Test Set (R² = {test_r2:.4f})')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('actual_vs_predicted.png', dpi=100, bbox_inches='tight')
plt.show()
print("✅ Actual vs Predicted plot created!")

## 8. Save Model and Scaler

In [ ]:
# Save model and scaler for deployment
print("Saving model artifacts...")
joblib.dump(model, 'model.pkl')
joblib.dump(scaler, 'scaler.pkl')

# Also save metadata
metadata = {
    'model_type': 'RandomForestRegressor',
    'feature_names': feature_names,
    'n_features': len(feature_names),
    'test_metrics': {
        'mse': float(test_mse),
        'rmse': float(test_rmse),
        'mae': float(test_mae),
        'r2': float(test_r2)
    }
}

with open('model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print("✅ Model artifacts saved!")
print("   - model.pkl")
print("   - scaler.pkl")
print("   - model_metadata.json")

## 9. Create Prediction Function

In [ ]:
def predict_sample(features: List[float]) -> Dict:
    """
    Make prediction for a single sample
    
    Args:
        features: List of 10 features
    
    Returns:
        Dictionary with prediction and metadata
    """
    if len(features) != 10:
        return {"error": f"Expected 10 features, got {len(features)}"}
    
    X_scaled = scaler.transform([features])
    prediction = model.predict(X_scaled)[0]
    
    return {
        "prediction": float(prediction),
        "features_count": len(features),
        "model_type": "RandomForestRegressor"
    }

# Test the prediction function
test_features = [0.05, -0.05, 0.03, -0.02, 0.01, -0.04, 0.02, -0.03, 0.04, -0.01]
result = predict_sample(test_features)
print(f"\n✅ Test Prediction:")
print(json.dumps(result, indent=2))

## 10. Flask API Deployment

### Option 1: Run Flask Locally
Run the following command in a terminal:
```bash
python flask_app.py
```
Then access: http://localhost:5000

### Option 2: Deploy to Heroku
See deployment guide in README.md

In [ ]:
# Flask API code (run separately with: python flask_app.py)
flask_code = '''
from flask import Flask, request, jsonify
import numpy as np
import joblib

app = Flask(__name__)
model = joblib.load('model.pkl')
scaler = joblib.load('scaler.pkl')

@app.route('/predict', methods=['POST'])
def predict():
    data = request.json
    features = np.array(data['features']).reshape(1, -1)
    X_scaled = scaler.transform(features)
    prediction = model.predict(X_scaled)[0]
    return jsonify({'prediction': float(prediction)})

@app.route('/health', methods=['GET'])
def health():
    return jsonify({'status': 'healthy'})

if __name__ == '__main__':
    app.run(debug=True)
'''

print("✅ Flask API code ready!")
print("Run: python flask_app.py")
print("API Endpoint: http://localhost:5000/predict")

## 11. FastAPI Deployment

Run the following command in a terminal:
```bash
uvicorn fastapi_app:app --reload
```
Then access: http://localhost:8000/docs

In [ ]:
# FastAPI code (run separately with: uvicorn fastapi_app:app --reload)
fastapi_code = '''
from fastapi import FastAPI
from pydantic import BaseModel
import numpy as np
import joblib

app = FastAPI(title="ML Prediction API")
model = joblib.load('model.pkl')
scaler = joblib.load('scaler.pkl')

class PredictionRequest(BaseModel):
    features: list

class PredictionResponse(BaseModel):
    prediction: float

@app.post("/predict", response_model=PredictionResponse)
def predict(request: PredictionRequest):
    features = np.array(request.features).reshape(1, -1)
    X_scaled = scaler.transform(features)
    prediction = model.predict(X_scaled)[0]
    return PredictionResponse(prediction=float(prediction))

@app.get("/health")
def health():
    return {"status": "healthy"}
'''

print("✅ FastAPI code ready!")
print("Run: uvicorn fastapi_app:app --reload")
print("API Docs: http://localhost:8000/docs")

## 12. Summary and Results

In [ ]:
print("\n" + "="*70)
print("🎉 END-TO-END DATA SCIENCE PROJECT COMPLETE! 🎉")
print("="*70)

print("\n📊 PROJECT SUMMARY:")
print(f"   Dataset: Diabetes (Regression)")
print(f"   Samples: {len(X)}")
print(f"   Features: {len(feature_names)}")
print(f"   Train/Test Split: 80/20")

print(f"\n🤖 MODEL: Random Forest Regressor")
print(f"   Estimators: 100")
print(f"   Max Depth: 15")

print(f"\n📈 PERFORMANCE (Test Set):")
print(f"   R² Score: {test_r2:.4f} ✅")
print(f"   RMSE: {test_rmse:.4f}")
print(f"   MAE: {test_mae:.4f}")

print(f"\n💾 SAVED ARTIFACTS:")
print(f"   ✓ model.pkl (trained model)")
print(f"   ✓ scaler.pkl (feature scaler)")
print(f"   ✓ model_metadata.json (metadata)")

print(f"\n📡 API OPTIONS:")
print(f"   1. Flask: python flask_app.py (Port 5000)")
print(f"   2. FastAPI: uvicorn fastapi_app:app --reload (Port 8000)")

print(f"\n🚀 DEPLOYMENT READY!")
print("="*70 + "\n")